In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import numpy as np
import os
import re
from load_results import abbrev_map_strategy

from matplotlib.backends.backend_pdf import PdfPages

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from color import darken_color, base_colors, saturate_color, vivid_color
from load_results import load, format, abbrev_map_score,abbrev_map_model,abbrev_map_task
import os

import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, cophenet
from scipy.stats import pearsonr
from itertools import combinations
import matplotlib.pyplot as plt
import itertools
import matplotlib.pyplot as plt


import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 220
df = load()
df = format(df)
# df = df.drop(columns=["selection_strategy_name_a"])

In [ ]:
display(df[df["score"].isna()].groupby(["model", "Experiment"]).count())


In [ ]:
offset_base_colors = base_colors[1:] + base_colors[:1]

vivid_colors = tuple(
    vivid_color(c, amount=1.6)   
    for c in offset_base_colors
)

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=vivid_colors)

In [ ]:
from plotting_config import scores_to_plot

In [ ]:
def build_dist_matrix_condensed(df_, reps=None):
    df_sym = pd.concat([
        df_,
        df_.rename(columns={"Rep_A": "Rep_B", "Rep_B": "Rep_A"})
    ], ignore_index=True)

    # 2. get reps from THIS group only
    if reps is None:
        reps = sorted(set(df_sym["Rep_A"]).union(df_sym["Rep_B"]))

    # 3. create diagonal for THIS group
    df_diag = pd.DataFrame({
        "Rep_A": reps,
        "Rep_B": reps,
        "score": 1.0
    })
    
    # 4. add diagonal
    df_full = pd.concat([df_sym, df_diag], ignore_index=True)
    
    cka_mat = (
        df_full
        .pivot_table(
            index="Rep_A",
            columns="Rep_B",
            values="score",
            aggfunc="mean" # no-op
        )
    )
    
    cka_mat = cka_mat.reindex(index=reps, columns=reps)
    cka_mat =  np.clip(cka_mat, 0.0, 1.0)
    dist = 1 - cka_mat
    
    return  squareform(dist, checks=False), cka_mat.index, cka_mat



if_sorting_strategies = [
    "all",
    "most_harmful",
    "most_helpful",
    "least_influential",
    "most_influential",
]

for (score_name, k, normalize), df__ in df.groupby(["score_name", "k", "normalize"]):
    if score_name not in scores_to_plot:
        continue
    for sorting_strategy in if_sorting_strategies:
        if score_name == "cka" and sorting_strategy != "all":
            continue
        
        print("score_name", score_name, k, normalize, sorting_strategy)
        if sorting_strategy == "all":
            df_ = df__
        else:
            df_ = df__[((df__["selection_strategy_name_a"] == sorting_strategy)
                                    | (df__["selection_strategy_name_a"] == "-")
                                )
                                &
                                (
                                    (df__["selection_strategy_name_b"] == sorting_strategy)
                                    | (df__["selection_strategy_name_b"] == "-")
                                )]

        if len(df_) == 0:
            continue
        dist_mat, labels, _ = build_dist_matrix_condensed(df_)
        Z = linkage(dist_mat, method="average", optimal_ordering=True)


        width_pt = 219.08614 
        
        # print(len(labels) )
        if len(labels) == 12:
            width_pt *= 2/3 

        if len(labels) == 12:
            aspect = 1.5 / 4
        else:
            aspect = 1 / 4
        width_in = width_pt / 72
        height_in = width_in * aspect

        fig, ax = plt.subplots(figsize=(width_in, height_in), constrained_layout=True)

        FONTSIZE = 4

        dendrogram(
            Z,
            labels=labels,
            leaf_rotation=45,
            leaf_font_size=FONTSIZE,
            ax=ax
        )

        path =  f"./figures/experiment-2/{sorting_strategy.replace('_','-')}/{score_name.replace('_','-')}-{k}-{normalize}.pdf"
        os.makedirs(os.path.dirname(path), exist_ok=True)

        ax.tick_params(axis='y', labelsize=FONTSIZE)
        ax.tick_params(axis='x', pad=0)
        if len(labels) == 12:
            ax.set_ylim(0, 1.05) 
        plt.savefig(
            path,
            format="pdf"

        )
        plt.show()


        plt.close(fig)


In [ ]:
def build_cophenetic(condensed):
    # print("condensed",condensed)
    Z = linkage(condensed, method="average", optimal_ordering=True)
    _, coph = cophenet(Z, condensed)
    return coph

In [ ]:
def display_score_groups(df_agg, pivot_columns):
    df_agg_ = df_agg.reset_index()

    # pivot using original column name
    if pivot_columns != "coss_score":
        pivot = df_agg_.pivot_table(
            index="score_name",
            columns=pivot_columns,
            values="cophenetic_corr"
        )
    else: 
        pivot = df_agg

    # now rename for display
    pivot = pivot.rename(columns=lambda x: x)
    pivot.index.name = "Metric \\rotatebox[origin=c]{270}{$\Rsh$}"
    pivot.columns.name = "Group $\\rightarrow$"


    pivot.index = [abbrev_map_score[row] if row in abbrev_map_score else row for row in pivot.index]
    pivot.columns = [f"\\rotatebox{{45}}{{{abbrev_map_model[col]}}}" if col in abbrev_map_model else col for col in pivot.columns]
    pivot.columns = [abbrev_map_task[col] if col in abbrev_map_task else col for col in pivot.columns]
    pivot.columns = [abbrev_map_score[col] if col in abbrev_map_score else col for col in pivot.columns]

    path = f"./figures/experiment-2/corr/{str(pivot_columns).split('_')[0]}.tex"
    print(path)

    styled = (
        pivot.style
        .background_gradient(cmap="viridis", axis=1 if pivot_columns != "coss_score" else None,).highlight_null(props="background-color: white; color: white;")
    ).format(precision=2)#.format_index(escape="latex", axis=1)

    display(styled)
    latex = styled.to_latex(
        convert_css=True,
        hrules=True,
        multicol_align="l",
        column_format='l' + 'l' * len(pivot.columns)
    )

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(latex)

In [ ]:
results = []
for (score_name, k, normalize, test_type), g in df[(~df["normalize"]) & (df["score_name"].isin(scores_to_plot))].groupby(["score_name", "k", "normalize", "test_type"]):
    
    
    structures = {}
    global_reps = sorted(
    set().union(*[
        set(g["Rep_A"]).union(set(g["Rep_B"]))
        ])
    )
    for name, sub in g.groupby(["Experiment","base_model"]):
        print(score_name, k, normalize, test_type, name)
        condensed, labels, _ = build_dist_matrix_condensed(sub, reps=global_reps) 
        structures[name] = (labels, build_cophenetic(condensed)) # dendrogram for one model (base-model + task)
    
    for (experiment_a,base_model_a), (experiment_b,base_model_b) in combinations(structures.keys(), 2):

            lab1, coph1 = structures[(experiment_a,base_model_a)]
            lab2, coph2 = structures[(experiment_b,base_model_b)]



            x = coph1
            y = coph2
            corr = pearsonr(x, y)[0]

            results.append({
                "score_name": score_name,
                "Experiment_A": experiment_a,
                "Experiment_B": experiment_b,
                "base-model_A": base_model_a,
                "base-model_B": base_model_b,
                "score_name": score_name,
                "k": k,
                "normalize": normalize,
                "cophenetic_corr": corr,
                "test_type": test_type 
            })
df_cophenetic = pd.DataFrame(results)

$\bar{\rho}_{d}=\frac{1}{|\mathcal{D}|\binom{|\mathcal{B}|}{2}}\sum_{d \in \mathcal{D}}\sum_{b_1 < b_2}\rho_{\mathrm{coph}}\!\big(C^{d,b_1}, C^{d,b_2}\big)$

In [ ]:
df_agg = df_cophenetic[(df_cophenetic["Experiment_A"] == df_cophenetic["Experiment_B"])].groupby(["score_name", "k", "normalize", "Experiment_A"]).mean(numeric_only=True)
# display(df_agg)
display_score_groups(df_agg,pivot_columns = "Experiment_A")

$\bar{\rho}_{b}=\frac{1}{|\mathcal{B}|\binom{|\mathcal{D}|}{2}}\sum_{b \in \mathcal{B}}\sum_{d_1 < d_2}\rho_{\mathrm{coph}}\!\big(C^{d_1,b}, C^{d_2,b}\big)$

In [ ]:
df_agg = df_cophenetic[(df_cophenetic["base-model_A"] == df_cophenetic["base-model_B"])].groupby(["score_name", "k", "normalize", "base-model_A"]).mean(numeric_only=True)
# display(df_agg)
display_score_groups(df_agg,pivot_columns = "base-model_A")

$\bar{\rho}_{d,b}=\frac{1}{\binom{|\mathcal{D}|}{2}\binom{|\mathcal{B}|}{2}}\sum_{d_1 < d_2}\sum_{b_1 < b_2}\rho_{\mathrm{coph}}\!\big(C^{d_1,b_1}, C^{d_2,b_2}\big).$

In [ ]:
df_marginal = df_cophenetic[(df_cophenetic["base-model_A"] != df_cophenetic["base-model_B"]) & (df_cophenetic["Experiment_A"] != df_cophenetic["Experiment_B"])].groupby(["score_name", "k", "normalize"]).mean(numeric_only=True)

def write_raw_latex(row, out_dir, col ):
    os.makedirs(out_dir, exist_ok=True)
    score = re.sub(r"[^a-zA-Z0-9]+", "", str(row.name))

    value = row[col]

    file_path = os.path.join(out_dir, f"{score}.tex")

    # raw LaTeX content (just the number)
    with open(file_path, "w") as f:
        content = f"{value:.2f}"
        print(file_path, "-->",content)
        f.write(content)

    return None  

df_marginal.apply(lambda row : write_raw_latex(row, out_dir = "latex_commands/joint_average_cophenetic_stability", col="cophenetic_corr"), axis=1)
""

In [ ]:
# drop k to enable comparison of cknna (k=25) with others (no k) 

df = df[( (df["score_name"].isin(['pw_acc', 'cka', 'rbo', 'spearman_rho', 'kendall']) & (df["k"] == -1))
           |
            ((df["score_name"] == "cknna") & (df["k"] == 25)))]
df = df.drop(columns=["k"])

In [ ]:
# as above but different grouping level:
# need to align func and rep representation names (most_inf, least_help are only applicable to func)
results = []
for sorting_strategy, sorting_strategy_abbrv in abbrev_map_strategy.items(): 
    for normalize, g in df[(~df["normalize"]) & (df["score_name"].isin(scores_to_plot))].groupby("normalize"):
        
        structures = {}
        align_func_rep_settings = lambda x: x.replace(sorting_strategy,"")
        g.Rep_A = g.Rep_A.apply(align_func_rep_settings)
        g.Rep_B = g.Rep_B.apply(align_func_rep_settings)
        
        blocked = "|".join([v.replace("$","") for v in abbrev_map_strategy.values()])

        g = g[
            (~g["Rep_A"].astype(str).str.contains(blocked, na=False)) &
            (~g["Rep_B"].astype(str).str.contains(blocked, na=False))
        ]
   
        group_reps = []
        
        for _, g_ in g.groupby(["score_name", "Experiment", "base_model"]):
            reps = set(g_["Rep_A"]).union(set(g_["Rep_B"]))
            group_reps.append(reps)
        global_reps = sorted(set.intersection(*group_reps))

        for name, sub in g.groupby(["score_name", "Experiment","base_model"]):
            condensed, labels, _ = build_dist_matrix_condensed(sub, reps=global_reps) 
            structures[name] = (labels, build_cophenetic(condensed)) 
        for (score_name_a, experiment_a,base_model_a), (score_name_b,experiment_b,base_model_b) in combinations(structures.keys(), 2):

                lab1, coph1 = structures[(score_name_a, experiment_a,base_model_a)]
                lab2, coph2 = structures[(score_name_b, experiment_b,base_model_b)]

                common = sorted(set(lab1) & set(lab2))
                i1 = {x: i for i, x in enumerate(lab1)}
                i2 = {x: i for i, x in enumerate(lab2)}

                x = coph1[[i1[c] for c in common]]
                y = coph2[[i2[c] for c in common]]

                corr = pearsonr(x, y)[0]

                results.append({
                    "score_name_A": score_name_a,
                    "score_name_B": score_name_b,
                    "Experiment_A": experiment_a,
                    "Experiment_B": experiment_b,
                    "base-model_A": base_model_a,
                    "base-model_B": base_model_b,
                    "normalize": normalize,
                    "cophenetic_corr": corr,
                    "sorting_strategy": sorting_strategy
                })
df_cophenetic = pd.DataFrame(results)

In [ ]:
result = (
    df_cophenetic[(df_cophenetic["base-model_A"] != df_cophenetic["base-model_B"]) & (df_cophenetic["Experiment_A"] != df_cophenetic["Experiment_B"])]
    .groupby(["sorting_strategy","score_name_A", "score_name_B",  "normalize", ])
    .mean(numeric_only=True)
)

matrix = result["cophenetic_corr"].unstack("score_name_B")

In [ ]:
display_score_groups(matrix.xs(
                        ("most_helpful", False),
                        level=("sorting_strategy", "normalize")
                        )
                     , pivot_columns="coss_score")

In [ ]:
display_score_groups(matrix.xs(
                        ("most_helpful", False),
                        level=("sorting_strategy", "normalize")
                        )
                     , pivot_columns="coss_score")

In [ ]:
matrix

## Choice of representations for case study

In [ ]:
for (score_name, k, normalize), df_ in df.groupby(["score_name", "k", "normalize"]):

    if score_name not in scores_to_plot:
        continue


    _, labels, cka_mat = build_dist_matrix_condensed(df_)

    labels = np.array(labels)
    cka = cka_mat.values

    n = 4
    n = min(n, len(labels))


    selected = [np.argmin((1 - cka).sum(axis=1))]

    for _ in range(n - 1):

        sim_to_selected = np.max(cka[:, selected], axis=1)
        diversity = 1 - sim_to_selected

        diversity[selected] = -1  # prevent duplicates

        next_idx = np.argmax(diversity)
        selected.append(next_idx)

    selected = np.array(selected)

    print(f"Selection for {score_name, k, normalize}:")
    print(labels[selected])
    break